In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import SGDClassifier, LogisticRegression

from category_encoders import TargetEncoder

# =========================================================
# Config
# =========================================================

PROJECT_ROOT = Path("/mnt/c/dev/my_ml_project")

DATA_DIR = PROJECT_ROOT / "data"
OOF_DIR = PROJECT_ROOT / "oof_preds"
SUB_DIR = PROJECT_ROOT / "submissions"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

TARGET = "임신 성공 여부"

FOLD_SEED = 42
N_SPLITS = 5
POS_WEIGHT = 190123 / 66228

SAVE_DIR = OOF_DIR / "combo_te_v1_linear_seed42"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SUB_SAVE_DIR = SUB_DIR / "linear_blend"
SUB_SAVE_DIR.mkdir(parents=True, exist_ok=True)

CHAMPION_4SEED_OOF = 0.7407705074934163
MLP_BEST_OOF = 0.74091342083597  # champion + mlp_avg_seed42_2024 w0.21

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH exists:", TRAIN_PATH.exists())
print("SAVE_DIR:", SAVE_DIR)

PROJECT_ROOT: /mnt/c/dev/my_ml_project
TRAIN_PATH exists: True
SAVE_DIR: /mnt/c/dev/my_ml_project/oof_preds/combo_te_v1_linear_seed42


In [3]:
def data_preprocessing(df):
    df = df.copy()
    time_cols = [
        '임신 시도 또는 마지막 임신 경과 연수',
        '난자 해동 경과일',
        '난자 혼합 경과일',
        '배아 이식 경과일',
        '배아 해동 경과일'
    ]

    for col in time_cols:
        df[f'{col}_performed'] = (
            df[col].notnull()
        ).astype(int)


    df = df.fillna(0)
    infertility_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 남성 요인',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    male_cols = [
        '불임 원인 - 남성 요인',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    female_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증'
    ]

    df['불임원인_총개수'] = df[infertility_cols].sum(axis=1)

    df['남성_원인_수'] = df[male_cols].sum(axis=1)

    df['여성_원인_수'] = df[female_cols].sum(axis=1)

    df['남녀_복합_원인'] = (
        (df['남성_원인_수'] > 0) &
        (df['여성_원인_수'] > 0)
    ).astype(int)

    df['원인불명'] = (
        df['불임원인_총개수'] == 0
    ).astype(int)

    count_cols = [
        '총 시술 횟수',
        'IVF 시술 횟수',
        'DI 시술 횟수',
        '총 임신 횟수',
        'IVF 임신 횟수',
        'DI 임신 횟수',
        '총 출산 횟수',
        'IVF 출산 횟수',
        'DI 출산 횟수',
        '클리닉 내 총 시술 횟수'
    ]

    count_map = {
        '0회': 0,
        '1회': 1,
        '2회': 2,
        '3회': 3,
        '4회': 4,
        '5회': 5,
        '6회 이상': 6,
    }

    for col in count_cols:
        df[col] = df[col].map(count_map).astype(float)

    df['고령여부'] = df['시술 당시 나이'].isin([
        '만38-39세',
        '만40-42세',
        '만43-44세',
        '만45-50세'
    ]).astype(int)


    df['배아_생성률'] = np.where(
        df['혼합된 난자 수'] == 0,
        0,
        df['총 생성 배아 수'] / df['혼합된 난자 수']
    )

    # 2. 배아 이식 효율
    df['배아_이식률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['이식된 배아 수'] / df['총 생성 배아 수']
    )

    # 3. 배아 냉동 비율
    df['배아_냉동률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['저장된 배아 수'] / df['총 생성 배아 수']
    )
    df['IVF_임신성공률'] = np.where(
        df['IVF 시술 횟수'] == 0,
        0,
        df['IVF 임신 횟수'] / df['IVF 시술 횟수']
    )

    df['DI_임신성공률'] = np.where(
        df['DI 시술 횟수'] == 0,
        0,
        df['DI 임신 횟수'] / df['DI 시술 횟수']
    )

    df['고령_난자수_interaction'] = (
        df['고령여부'] *
        df['수집된 신선 난자 수']
    )

    df['배아이식_수행여부'] = (
        df['이식된 배아 수'] > 0
    ).astype(int)

    df['배아_이식_집중도'] = np.where(
        (df['이식된 배아 수'] + df['저장된 배아 수']) == 0,
        0,
        df['이식된 배아 수'] /
        (
            df['이식된 배아 수'] +
            df['저장된 배아 수']
        )
    )
    df["배아 생성 주요 이유"] = (
        df["배아 생성 주요 이유"]
        .astype(str)
        .astype("category")
    )
    df["특정 시술 유형"] = (
        df["특정 시술 유형"]
        .astype(str)
        .astype("category")
    )

    # 고령 × 이식 배아 수
    df['고령_배아이식'] = (
        df['고령여부'] *
        df['이식된 배아 수']
    )

    # 고령 × 총 생성 배아 수
    df['고령_배아생성'] = (
        df['고령여부'] *
        df['총 생성 배아 수']
    )

    # 고령 × 저장 배아 수
    df['고령_배아저장'] = (
        df['고령여부'] *
        df['저장된 배아 수']
    )

    # 고령 × 미세주입 난자 수
    df['고령_미세주입난자'] = (
        df['고령여부'] *
        df['미세주입된 난자 수']
    )

    df['출산_임신_전환율'] = np.where(
        df['총 임신 횟수'] == 0,
        0,
        df['총 출산 횟수'] / df['총 임신 횟수']
    )

    df['클리닉_집중도'] = np.where(
        df['총 시술 횟수'] == 0,
        0,
        df['클리닉 내 총 시술 횟수'] / df['총 시술 횟수']
    )

    df['첫_시술_여부'] = (
        df['총 시술 횟수'] == 0
    ).astype(int)



    binary_keywords = [
        "코드", "나이", "유형", "여부", "원인", "이유", "횟수", "출처"
    ]

    binary_cols = [
        col for col in df.columns
        if any(keyword in col for keyword in binary_keywords)
    ]

    df[binary_cols] = df[binary_cols].astype('category')

    object_cols = df.select_dtypes(include="object").columns

    df[object_cols] = df[object_cols].astype(str)

    cat_cols = df.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in cat_cols:
        if col != TARGET:
            df[col] = df[col].astype(str)

    drop_cols = [
        '배아이식_수행여부'
    ]
    df = df.drop(
        columns=drop_cols
        )
    return df

In [4]:
def rank01(pred):
    return rankdata(pred) / len(pred)

In [5]:
def add_combo_columns(X):
    X = X.copy()

    combo_pairs = [
        ("시술 당시 나이", "난자 출처"),
        ("시술 당시 나이", "정자 출처"),
        ("시술 당시 나이", "시술 유형"),
        ("시술 당시 나이", "특정 시술 유형"),
        ("시술 유형", "난자 출처"),
        ("시술 유형", "정자 출처"),
        ("특정 시술 유형", "난자 출처"),
        ("특정 시술 유형", "정자 출처"),
        ("난자 출처", "정자 출처"),
        ("배란 유도 유형", "시술 당시 나이"),
    ]

    combo_cols = []

    for col1, col2 in combo_pairs:
        if col1 in X.columns and col2 in X.columns:
            new_col = f"{col1}_{col2}_combo"
            X[new_col] = X[col1].astype(str) + "_" + X[col2].astype(str)
            combo_cols.append(new_col)

    return X, combo_cols


def add_oof_target_encoding(X, y, cols, n_splits=5, smoothing=10, random_state=42):
    X = X.copy().reset_index(drop=True)
    y = y.astype(int).reset_index(drop=True)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    te_features = pd.DataFrame(index=X.index)

    for col in cols:
        te_features[f"{col}_TE"] = 0.0

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"Target Encoding Fold {fold}")

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_tr = y.iloc[train_idx]

        encoder = TargetEncoder(
            cols=cols,
            smoothing=smoothing
        )

        encoder.fit(X_tr[cols], y_tr)
        encoded_val = encoder.transform(X_val[cols])

        for col in cols:
            te_features.loc[val_idx, f"{col}_TE"] = encoded_val[col].values

    final_encoder = TargetEncoder(
        cols=cols,
        smoothing=smoothing
    )

    final_encoder.fit(X[cols], y)

    X_te = pd.concat([X, te_features], axis=1)

    return X_te, final_encoder

In [6]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
submission = pd.read_csv(SUBMISSION_PATH)

X_raw = train_raw.drop(columns=[TARGET])
y = train_raw[TARGET].astype(int).reset_index(drop=True)
X_test_raw = test_raw.copy()

id_cols = [col for col in X_raw.columns if "ID" in col.upper()]

X_raw = X_raw.drop(columns=id_cols, errors="ignore").reset_index(drop=True)
X_test_raw = X_test_raw.drop(columns=id_cols, errors="ignore").reset_index(drop=True)

# 기존 champion preprocessing 사용
X = data_preprocessing(X_raw)
X_test = data_preprocessing(X_test_raw)

X_combo, combo_cols = add_combo_columns(X)
X_test_combo, _ = add_combo_columns(X_test)

base_te_cols = [
    "시술 시기 코드",
    "시술 유형",
    "특정 시술 유형",
    "배란 유도 유형",
    "난자 출처",
    "정자 출처",
    "배아 생성 주요 이유",
    "시술 당시 나이",
]

base_te_cols = [col for col in base_te_cols if col in X_combo.columns]
te_cols = base_te_cols + combo_cols

print("base_te_cols:", base_te_cols)
print("combo_cols:", combo_cols)
print("te_cols count:", len(te_cols))

X_te_combo, te_encoder_combo = add_oof_target_encoding(
    X_combo,
    y,
    cols=te_cols,
    n_splits=N_SPLITS,
    smoothing=10,
    random_state=FOLD_SEED
)

test_te_values = te_encoder_combo.transform(X_test_combo[te_cols])

for col in te_cols:
    X_test_combo[f"{col}_TE"] = test_te_values[col].values

# combo 문자열 원본 제거
X_te_combo = X_te_combo.drop(columns=combo_cols, errors="ignore")
X_test_te_combo = X_test_combo.drop(columns=combo_cols, errors="ignore")

# 컬럼 정렬
X_test_te_combo = X_test_te_combo[X_te_combo.columns]

X_stack = X_te_combo.copy().reset_index(drop=True)
X_test_stack = X_test_te_combo.copy().reset_index(drop=True)
y_stack = y.reset_index(drop=True)

print("X_stack:", X_stack.shape)
print("X_test_stack:", X_test_stack.shape)
print("y_stack:", y_stack.shape)

assert list(X_stack.columns) == list(X_test_stack.columns)
assert len(X_stack) == len(y_stack)
assert len(X_test_stack) == len(submission)

base_te_cols: ['시술 시기 코드', '시술 유형', '특정 시술 유형', '배란 유도 유형', '난자 출처', '정자 출처', '배아 생성 주요 이유', '시술 당시 나이']
combo_cols: ['시술 당시 나이_난자 출처_combo', '시술 당시 나이_정자 출처_combo', '시술 당시 나이_시술 유형_combo', '시술 당시 나이_특정 시술 유형_combo', '시술 유형_난자 출처_combo', '시술 유형_정자 출처_combo', '특정 시술 유형_난자 출처_combo', '특정 시술 유형_정자 출처_combo', '난자 출처_정자 출처_combo', '배란 유도 유형_시술 당시 나이_combo']
te_cols count: 18
Target Encoding Fold 1
Target Encoding Fold 2
Target Encoding Fold 3
Target Encoding Fold 4
Target Encoding Fold 5
X_stack: (256351, 110)
X_test_stack: (90067, 110)
y_stack: (256351,)


In [7]:
def make_linear_preprocessor(X):
    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    print("numeric_features:", len(numeric_features))
    print("categorical_features:", len(categorical_features))

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=20,
                dtype=np.float32,
            )
        ),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
        sparse_threshold=1.0,
    )

    return preprocessor

In [9]:
LINEAR_CONFIGS = [
    {
        "name": "sgd_log_l2_a1e4",
        "model": SGDClassifier(
            loss="log_loss",
            penalty="l2",
            alpha=1e-4,
            max_iter=2000,
            tol=1e-4,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )
    },
    {
        "name": "sgd_log_l2_a3e5",
        "model": SGDClassifier(
            loss="log_loss",
            penalty="l2",
            alpha=3e-5,
            max_iter=2000,
            tol=1e-4,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )
    },
    {
        "name": "sgd_log_elastic_a1e4_l1r005",
        "model": SGDClassifier(
            loss="log_loss",
            penalty="elasticnet",
            alpha=1e-4,
            l1_ratio=0.05,
            max_iter=2000,
            tol=1e-4,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )
    },

    {
        "name": "logreg_saga_C1",
        "model": LogisticRegression(
            C=1.0,
            penalty="l2",
            solver="saga",
            max_iter=1000,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42,
            verbose=0,
        )
    },
]

In [10]:
def get_model_score(model, X):
    """
    AUC용 score 추출.
    predict_proba가 있으면 양성 확률,
    없으면 decision_function 사용.
    """
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "decision_function"):
        return model.decision_function(X)

    raise ValueError("Model has neither predict_proba nor decision_function")


def train_linear_oof_test(
    X_stack,
    X_test_stack,
    y_stack,
    config,
    n_splits=5,
    fold_seed=42,
    save_dir=None,
):
    name = config["name"]
    base_model = config["model"]

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=fold_seed
    )

    oof = np.zeros(len(X_stack))
    test_pred = np.zeros(len(X_test_stack))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print("\n" + "=" * 100)
        print(f"Linear Model: {name} / Fold {fold}")
        print("=" * 100)

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx]
        y_val = y_stack.iloc[val_idx]

        preprocessor = make_linear_preprocessor(X_tr)

        X_tr_mat = preprocessor.fit_transform(X_tr)
        X_val_mat = preprocessor.transform(X_val)
        X_test_mat = preprocessor.transform(X_test_stack)

        print("X_tr_mat:", X_tr_mat.shape)
        print("X_val_mat:", X_val_mat.shape)
        print("X_test_mat:", X_test_mat.shape)

        # clone 대신 deepcopy 사용
        model = copy.deepcopy(base_model)

        model.fit(X_tr_mat, y_tr)

        val_pred = get_model_score(model, X_val_mat)
        test_fold_pred = get_model_score(model, X_test_mat)

        fold_auc = roc_auc_score(y_val, val_pred)
        print(f"{name} Fold {fold} AUC:", fold_auc)

        oof[val_idx] = val_pred
        test_pred += test_fold_pred / n_splits
        fold_scores.append(fold_auc)

        if save_dir is not None:
            np.save(save_dir / f"{name}_partial_oof_fold{fold}.npy", oof)
            np.save(save_dir / f"{name}_partial_test_fold{fold}.npy", test_pred)

    oof_auc = roc_auc_score(y_stack, oof)

    print("\n" + "=" * 100)
    print("Linear Result:", name)
    print("fold_scores:", fold_scores)
    print("mean fold auc:", np.mean(fold_scores))
    print("OOF AUC:", oof_auc)
    print("=" * 100)

    if save_dir is not None:
        np.save(save_dir / f"{name}_oof.npy", oof)
        np.save(save_dir / f"{name}_test_pred.npy", test_pred)

        pd.DataFrame([{
            "model": name,
            "fold_seed": fold_seed,
            "oof_auc": oof_auc,
            "mean_fold_auc": np.mean(fold_scores),
            "fold_scores": str(fold_scores),
            "config": str(base_model),
        }]).to_csv(
            save_dir / f"{name}_summary.csv",
            index=False
        )

    return oof, test_pred, fold_scores

In [11]:
import copy

linear_results = {}

for config in LINEAR_CONFIGS:
    name = config["name"]

    oof, test_pred, fold_scores = train_linear_oof_test(
        X_stack=X_stack,
        X_test_stack=X_test_stack,
        y_stack=y_stack,
        config=config,
        n_splits=N_SPLITS,
        fold_seed=FOLD_SEED,
        save_dir=SAVE_DIR,
    )

    linear_results[name] = {
        "oof": oof,
        "test_pred": test_pred,
        "oof_auc": roc_auc_score(y_stack, oof),
        "fold_scores": fold_scores,
    }

    print("\nSaved:", name)
    print("OOF:", linear_results[name]["oof_auc"])


Linear Model: sgd_log_l2_a1e4 / Fold 1
numeric_features: 56
categorical_features: 54
X_tr_mat: (205080, 256)
X_val_mat: (51271, 256)
X_test_mat: (90067, 256)
sgd_log_l2_a1e4 Fold 1 AUC: 0.7310136264326209

Linear Model: sgd_log_l2_a1e4 / Fold 2
numeric_features: 56
categorical_features: 54
X_tr_mat: (205081, 257)
X_val_mat: (51270, 257)
X_test_mat: (90067, 257)
sgd_log_l2_a1e4 Fold 2 AUC: 0.7339147890276038

Linear Model: sgd_log_l2_a1e4 / Fold 3
numeric_features: 56
categorical_features: 54
X_tr_mat: (205081, 255)
X_val_mat: (51270, 255)
X_test_mat: (90067, 255)
sgd_log_l2_a1e4 Fold 3 AUC: 0.7334107892797674

Linear Model: sgd_log_l2_a1e4 / Fold 4
numeric_features: 56
categorical_features: 54
X_tr_mat: (205081, 256)
X_val_mat: (51270, 256)
X_test_mat: (90067, 256)
sgd_log_l2_a1e4 Fold 4 AUC: 0.7317366513656243

Linear Model: sgd_log_l2_a1e4 / Fold 5
numeric_features: 56
categorical_features: 54
X_tr_mat: (205081, 257)
X_val_mat: (51270, 257)
X_test_mat: (90067, 257)
sgd_log_l2_a1e4 F

/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


logreg_saga_C1 Fold 1 AUC: 0.7325547225848041

Linear Model: logreg_saga_C1 / Fold 2
numeric_features: 56
categorical_features: 54
X_tr_mat: (205081, 257)
X_val_mat: (51270, 257)
X_test_mat: (90067, 257)


/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


logreg_saga_C1 Fold 2 AUC: 0.737631340173025

Linear Model: logreg_saga_C1 / Fold 3
numeric_features: 56
categorical_features: 54
X_tr_mat: (205081, 255)
X_val_mat: (51270, 255)
X_test_mat: (90067, 255)


...

/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


logreg_saga_C1 Fold 4 AUC: 0.7326122337635942

Linear Model: logreg_saga_C1 / Fold 5
numeric_features: 56
categorical_features: 54
X_tr_mat: (205081, 257)
X_val_mat: (51270, 257)
X_test_mat: (90067, 257)


/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


logreg_saga_C1 Fold 5 AUC: 0.7345362174843585

Linear Result: logreg_saga_C1
fold_scores: [0.7325547225848041, 0.737631340173025, 0.734004614893194, 0.7326122337635942, 0.7345362174843585]
mean fold auc: 0.7342678257797951
OOF AUC: 0.7342564189660454

Saved: logreg_saga_C1
OOF: 0.7342564189660454


/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [12]:
linear_summary = pd.DataFrame([
    {
        "name": name,
        "oof_auc": result["oof_auc"],
        "fold_scores": result["fold_scores"],
    }
    for name, result in linear_results.items()
]).sort_values("oof_auc", ascending=False)

display(linear_summary)
linear_summary.to_csv(SAVE_DIR / "linear_summary.csv", index=False)

,name,oof_auc,fold_scores
3,logreg_saga_C1,0.734256,"[0.7325547225848041, 0.737631340173025, 0.7340..."
2,sgd_log_elastic_a1e4_l1r005,0.730181,"[0.7291876862482795, 0.736774859281001, 0.7325..."
0,sgd_log_l2_a1e4,0.728563,"[0.7310136264326209, 0.7339147890276038, 0.733..."
1,sgd_log_l2_a3e5,0.722357,"[0.7256142705132822, 0.7338348908659911, 0.731..."


In [13]:
# Champion 4-seed OOF
seed42_oof = np.load(OOF_DIR / "combo_te_s10" / "final_combo_rank.npy")
seed2024_oof = np.load(OOF_DIR / "combo_te_v1_s10_seed2024" / "final_oof_seed2024.npy")
seed777_oof = np.load(OOF_DIR / "combo_te_v1_s10_seed777" / "final_oof_seed777.npy")
seed999_oof = np.load(OOF_DIR / "combo_te_v1_s10_seed999" / "final_oof_seed999.npy")

champion_4seed_oof = (
    seed42_oof +
    seed2024_oof +
    seed777_oof +
    seed999_oof
) / 4

champion_4seed_score = roc_auc_score(y_stack, champion_4seed_oof)

print("Champion 4-seed OOF:", champion_4seed_score)

Champion 4-seed OOF: 0.7407705074934163


In [14]:
# Champion 4-seed test
champion_4seed_test = np.load(
    SUB_DIR
    / "seed_ensemble"
    / "final_pred_combo_te_v1_seed42_2024_777_999_avg.npy"
)

print("champion_4seed_test:", champion_4seed_test.shape)

champion_4seed_test: (90067,)


In [15]:
# MLP avg seed42+2024
MLP2024_DIR = OOF_DIR / "combo_te_v1_mlp_seed2024"

mlp_avg_oof = np.load(MLP2024_DIR / "mlp_avg_seed42_2024_oof.npy")
mlp_avg_test = np.load(MLP2024_DIR / "mlp_avg_seed42_2024_test_pred.npy")

print("MLP avg OOF:", roc_auc_score(y_stack, mlp_avg_oof))

MLP avg OOF: 0.7387467936612893


In [16]:
w_mlp_best = 0.21

champ_mlp_oof = (
    (1 - w_mlp_best) * rank01(champion_4seed_oof)
    + w_mlp_best * rank01(mlp_avg_oof)
)

champ_mlp_test = (
    (1 - w_mlp_best) * rank01(champion_4seed_test)
    + w_mlp_best * rank01(mlp_avg_test)
)

champ_mlp_score = roc_auc_score(y_stack, champ_mlp_oof)

print("Champion + MLP avg w0.21 OOF:", champ_mlp_score)

Champion + MLP avg w0.21 OOF: 0.74091342083597


In [17]:
def search_base_plus_linear(
    y_true,
    base_oof,
    linear_oof,
    base_test,
    linear_test,
    base_name,
    linear_name,
    out_dir,
    max_w=0.30,
):
    base_score = roc_auc_score(y_true, base_oof)
    linear_score = roc_auc_score(y_true, linear_oof)

    base_rank_oof = rank01(base_oof)
    linear_rank_oof = rank01(linear_oof)

    best_score = base_score
    best_w = 0.0
    best_oof = base_oof.copy()

    rows = []

    for w_linear in np.arange(0.00, max_w + 0.001, 0.01):
        blend_oof = (
            (1 - w_linear) * base_rank_oof
            + w_linear * linear_rank_oof
        )

        score = roc_auc_score(y_true, blend_oof)

        rows.append({
            "base_name": base_name,
            "linear_name": linear_name,
            "w_linear": w_linear,
            "base_oof": base_score,
            "linear_oof": linear_score,
            "blend_oof": score,
            "improvement": score - base_score,
        })

        if score > best_score:
            best_score = score
            best_w = w_linear
            best_oof = blend_oof.copy()

    result_df = pd.DataFrame(rows).sort_values("blend_oof", ascending=False)

    result_df.to_csv(
        out_dir / f"blend_search_{base_name}_{linear_name}.csv",
        index=False
    )

    np.save(
        out_dir / f"best_oof_{base_name}_{linear_name}_w{best_w:.2f}.npy",
        best_oof
    )

    print("\n" + "=" * 80)
    print("base:", base_name)
    print("linear:", linear_name)
    print("base_oof:", base_score)
    print("linear_oof:", linear_score)
    print("best_blend_oof:", best_score)
    print("best_w_linear:", best_w)
    print("improvement:", best_score - base_score)
    print("=" * 80)

    return {
        "base_name": base_name,
        "linear_name": linear_name,
        "base_oof": base_score,
        "linear_oof": linear_score,
        "best_blend_oof": best_score,
        "best_w_linear": best_w,
        "improvement": best_score - base_score,
        "linear_test": linear_test,
        "base_test": base_test,
    }

In [18]:
blend_results = []

for name, result in linear_results.items():
    # A. champion 4seed + linear
    res_champ = search_base_plus_linear(
        y_true=y_stack,
        base_oof=champion_4seed_oof,
        linear_oof=result["oof"],
        base_test=champion_4seed_test,
        linear_test=result["test_pred"],
        base_name="champion4seed",
        linear_name=name,
        out_dir=SAVE_DIR,
        max_w=0.30,
    )
    blend_results.append(res_champ)

    # B. champion + MLP + linear
    res_mlp = search_base_plus_linear(
        y_true=y_stack,
        base_oof=champ_mlp_oof,
        linear_oof=result["oof"],
        base_test=champ_mlp_test,
        linear_test=result["test_pred"],
        base_name="champion4seed_mlpavg_w021",
        linear_name=name,
        out_dir=SAVE_DIR,
        max_w=0.30,
    )
    blend_results.append(res_mlp)


base: champion4seed
linear: sgd_log_l2_a1e4
base_oof: 0.7407705074934163
linear_oof: 0.7285631883486179
best_blend_oof: 0.7407705074934163
best_w_linear: 0.0
improvement: 0.0

base: champion4seed_mlpavg_w021
linear: sgd_log_l2_a1e4
base_oof: 0.74091342083597
linear_oof: 0.7285631883486179
best_blend_oof: 0.74091342083597
best_w_linear: 0.0
improvement: 0.0

base: champion4seed
linear: sgd_log_l2_a3e5
base_oof: 0.7407705074934163
linear_oof: 0.7223569976852808
best_blend_oof: 0.7407708109132076
best_w_linear: 0.01
improvement: 3.034197912921144e-07

base: champion4seed_mlpavg_w021
linear: sgd_log_l2_a3e5
base_oof: 0.74091342083597
linear_oof: 0.7223569976852808
best_blend_oof: 0.74091342083597
best_w_linear: 0.0
improvement: 0.0

base: champion4seed
linear: sgd_log_elastic_a1e4_l1r005
base_oof: 0.7407705074934163
linear_oof: 0.7301814761181911
best_blend_oof: 0.7407705074934163
best_w_linear: 0.0
improvement: 0.0

base: champion4seed_mlpavg_w021
linear: sgd_log_elastic_a1e4_l1r005
base

In [19]:
blend_summary = pd.DataFrame([
    {
        "base_name": r["base_name"],
        "linear_name": r["linear_name"],
        "base_oof": r["base_oof"],
        "linear_oof": r["linear_oof"],
        "best_blend_oof": r["best_blend_oof"],
        "best_w_linear": r["best_w_linear"],
        "improvement": r["improvement"],
        "beats_mlp_best": r["best_blend_oof"] - MLP_BEST_OOF,
    }
    for r in blend_results
]).sort_values("best_blend_oof", ascending=False)

display(blend_summary)
blend_summary.to_csv(SAVE_DIR / "linear_blend_summary.csv", index=False)

,base_name,linear_name,base_oof,linear_oof,best_blend_oof,best_w_linear,improvement,beats_mlp_best
1,champion4seed_mlpavg_w021,sgd_log_l2_a1e4,0.740913,0.728563,0.740913,0.00,0.000000e+00,0.000000
3,champion4seed_mlpavg_w021,sgd_log_l2_a3e5,0.740913,0.722357,0.740913,0.00,0.000000e+00,0.000000
7,champion4seed_mlpavg_w021,logreg_saga_C1,0.740913,0.734256,0.740913,0.00,0.000000e+00,0.000000
5,champion4seed_mlpavg_w021,sgd_log_elastic_a1e4_l1r005,0.740913,0.730181,0.740913,0.00,0.000000e+00,0.000000
2,champion4seed,sgd_log_l2_a3e5,0.740771,0.722357,0.740771,0.01,3.034198e-07,-0.000143
0,champion4seed,sgd_log_l2_a1e4,0.740771,0.728563,0.740771,0.00,0.000000e+00,-0.000143
4,champion4seed,sgd_log_elastic_a1e4_l1r005,0.740771,0.730181,0.740771,0.00,0.000000e+00,-0.000143
6,champion4seed,logreg_saga_C1,0.740771,0.734256,0.740771,0.00,0.000000e+00,-0.000143
